In [10]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

True

In [11]:
client = OpenAI()

In [12]:

with open("data/payment_utils.py") as f:
    payment_code = f.read()

with open("data/auth.py") as f:
    auth_code = f.read()

with open("data/team_standards.md") as f:
    team_standards = f.read()

In [13]:
type(payment_code)
type(auth_code)
type(team_standards)

str

In [14]:
MODEL = "gpt-4o-mini" 
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": f"Review this Python code for bugs and security issues:\n\n{payment_code}"
        }
    ]
)





In [15]:
# print(response.choices[0].message.content)

In [16]:
display(Markdown(response.choices[0].message.content))


The code you've shared has several potential bugs and security issues. Let's break them down:

### 1. SQL Injection Vulnerability
The `get_user` function constructs a SQL query using string interpolation, which is vulnerable to SQL injection attacks. If an attacker can control the `user_id` input, they could potentially execute arbitrary SQL commands.

**Fix: Use parameterized queries.**
```python
def get_user(user_id):
    import sqlite3
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    query = "SELECT * FROM users WHERE id = ?"
    cursor.execute(query, (user_id,))
    return cursor.fetchone()
```

### 2. Database Connection Management
The current implementation does not close the database connection, which can lead to resource leaks. It is also advisable to use a context manager to manage the connection, which ensures it gets closed properly.

**Fix: Use a context manager with `sqlite3`.**
```python
def get_user(user_id):
    import sqlite3
    with sqlite3.connect('users.db') as conn:
        cursor = conn.cursor()
        query = "SELECT * FROM users WHERE id = ?"
        cursor.execute(query, (user_id,))
        return cursor.fetchone()
```

### 3. Payment Processing Function
While `process_payment` is currently just a placeholder and vulnerable to issues if extended in the future, it's important to consider how sensitive payment information is handled. Storing or improperly using the card number can lead to severe security issues.

**Recommendations:**
- Never log sensitive information like card numbers.
- Ensure secure handling, authorization, and encryption practices are followed. 

For now, just avoid printing the card number.
```python
def process_payment(amount, card_number):
    # Log payment information securely, avoiding sensitive data exposure
    print(f"Processing {amount} payment.")
    return True
```

### 4. Discount Calculation
The `calculate_discount` function does not validate the inputs, which can lead to unexpected behavior or errors if negative values are provided.

**Fix: Add input validation.**
```python
def calculate_discount(price, discount):
    if price < 0 or discount < 0:
        raise ValueError("Price and discount must be non-negative.")
    return price - discount
```

### Summary of Fixed Code
Here’s the revised code with the outlined fixes:

```python
# data/payment_utils.py
def get_user(user_id):
    import sqlite3
    with sqlite3.connect('users.db') as conn:
        cursor = conn.cursor()
        query = "SELECT * FROM users WHERE id = ?"
        cursor.execute(query, (user_id,))
        return cursor.fetchone()

def process_payment(amount, card_number):
    # Log payment information securely, avoiding sensitive data exposure
    print(f"Processing {amount} payment.")
    return True

def calculate_discount(price, discount):
    if price < 0 or discount < 0:
        raise ValueError("Price and discount must be non-negative.")
    return price - discount
```

By applying these changes to your code, you improve its security, manageability, and robustness.

In [17]:

# THE CORE PROBLEM:
#   YOU are still doing all the work around the LLM:
#     → You decided which file to paste
#     → You read the output
#     → You go post the comment on GitHub manually
#     → You decide whether to block the PR
#     → You repeat this for every file, every PR, every day

# YOU ---> this need to replace ??


In [18]:

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": (
                "The SQL injection on line 14 of payment_utils.py is critical. "
                "Post a review comment on GitHub PR #247 blocking the merge."
            )
        }
    ]
)

display(Markdown(response.choices[0].message.content))


Certainly! Here's a review comment you can post on GitHub PR #247 regarding the SQL injection issue:

---

# Review Comment
## ⚠️ Critical Security Issue - SQL Injection Vulnerability

I have identified a critical SQL injection vulnerability present in `payment_utils.py` at line 14. This issue poses a significant security risk as it allows for unauthorized access and manipulation of the database. 

**Recommendation:**
To mitigate this risk, I strongly recommend using parameterized queries or prepared statements, which can help prevent SQL injection attacks. 

For example, rather than directly interpolating variables into the SQL query, use placeholders in the query and pass the parameters separately.

**Action Required:**
Please fix this vulnerability before merging the pull request. It is essential to ensure the security of our application and protect our user data.

Thank you!

--- 

Feel free to customize this comment as needed!

In [19]:

def read_file(filepath: str) -> str:
    """Read a file from the data directory."""
    try:
        full_path = f"data/{filepath}"
        with open(full_path) as f:
            return f.read()
    except FileNotFoundError:
        return f"ERROR: File '{filepath}' not found"

def post_review_comment(filepath: str, line: int, severity: str, comment: str) -> str:
    """Post an inline review comment on a PR. (Simulated — real GitHub API in Session 2)"""
    print(f"  [COMMENT POSTED] {filepath}:{line} [{severity.upper()}] {comment}")
    return f"Comment posted on {filepath} line {line}"

def set_pr_status(pr_number: int, status: str, reason: str) -> str:
    """Approve or block a PR from merging. (Simulated — real GitHub API in Session 2)"""
    emoji = "✅" if status == "approved" else "🚫"
    print(f"  [PR STATUS] PR #{pr_number} → {status.upper()} {emoji} | Reason: {reason}")
    return f"PR #{pr_number} set to {status}"



In [20]:
TOOLS = {
    "read_file": read_file,
    "post_review_comment": post_review_comment,
    "set_pr_status": set_pr_status
}

In [21]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a source code file from the repository",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {"type": "string", "description": "Filename to read, e.g. 'payment_utils.py'"}
                },
                "required": ["filepath"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "post_review_comment",
            "description": "Post an inline review comment on a specific line in a file",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath":  {"type": "string", "description": "File being commented on"},
                    "line":      {"type": "integer", "description": "Line number of the issue"},
                    "severity":  {"type": "string",  "description": "p0 (critical), p1 (high), p2 (medium)"},
                    "comment":   {"type": "string",  "description": "The review comment explaining the issue and fix"}
                },
                "required": ["filepath", "line", "severity", "comment"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "set_pr_status",
            "description": "Set the PR status to approved or blocked",
            "parameters": {
                "type": "object",
                "properties": {
                    "pr_number": {"type": "integer", "description": "The PR number"},
                    "status":    {"type": "string",  "description": "approved or blocked"},
                    "reason":    {"type": "string",  "description": "One line reason for the decision"}
                },
                "required": ["pr_number", "status", "reason"]
            }
        }
    }
]

In [22]:
SYSTEM_PROMPT = """You are CodeSentinel, an autonomous code review agent for RByte Pay.

Your goal: Review the PR thoroughly, post specific comments, and set a final status.

You have these tools:
- read_file: read any source file
- post_review_comment: post an inline comment with severity and fix
- set_pr_status: approve or block the PR

Rules:
- Always read team_standards.md first — every comment must reference a standard
- Review ALL changed files — do not stop after one
- Post specific comments with line numbers and exact fixes
- Severity: p0 = critical/block, p1 = high, p2 = medium
"""

In [ ]:
def run_agent(pr_number, changed_files):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Review PR #{pr_number}. "
                f"Changed files: {', '.join(changed_files)}. "
                f"Start by reading team standards, then review each file."
            )
        }
    ]

    step =0 
    max_steps = 15
    while step < max_steps:
        step += 1
        print(f"--- Step {step} ---")
        # THINK: ask the LLM what to do next
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_SCHEMAS,
            tool_choice="auto"   # LLM decides: call a tool OR give final answer
        )

        message = response.choices[0].message

        # DECIDE: did the LLM want to call a tool or is it done?
        if not message.tool_calls:
            # No tool call = LLM decided it is finished
            print("\nAgent done.")
            print("\nFinal message:")
            print(message.content)
            break

        # ACT: the LLM requested one or more tool calls — run them
        # Append the assistant's decision to history first
        messages.append(message)

        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"  Calling: {tool_name}({tool_args})")

            # Look up and run the actual Python function
            if tool_name in TOOLS:
                result = TOOLS[tool_name](**tool_args)
            else:
                result = f"ERROR: Unknown tool '{tool_name}'"

            # OBSERVE: feed the tool result back into the conversation
            # Without this, the LLM would not know what the tool returned
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })

    else:
        # Loop hit max_steps without finishing — this is a real problem
        # We will see this happen in Cell 9
        print(f"\nWARNING: Agent hit max_steps ({max_steps}) without finishing.")
        print("This is the first thing that breaks in production.") 

    

In [24]:
display(Markdown(
run_agent(
    pr_number=247,
    changed_files=["payment_utils.py", "auth.py"]
)))

--- Step 1 ---
  Calling: read_file({'filepath': 'team_standards.md'})
--- Step 2 ---
  Calling: read_file({'filepath': 'payment_utils.py'})
  Calling: read_file({'filepath': 'auth.py'})
--- Step 3 ---
  Calling: post_review_comment({'filepath': 'payment_utils.py', 'line': 5, 'severity': 'p1', 'comment': 'Line 5 has a SQL injection vulnerability. Use parameterized queries to prevent this issue (SQLAlchemy).'})
  [COMMENT POSTED] payment_utils.py:5 [P1] Line 5 has a SQL injection vulnerability. Use parameterized queries to prevent this issue (SQLAlchemy).
  Calling: post_review_comment({'filepath': 'payment_utils.py', 'line': 8, 'severity': 'p2', 'comment': 'Line 8 logs sensitive card number information. Avoid logging sensitive data.'})
  [COMMENT POSTED] payment_utils.py:8 [P2] Line 8 logs sensitive card number information. Avoid logging sensitive data.
  Calling: post_review_comment({'filepath': 'auth.py', 'line': 10, 'severity': 'p2', 'comment': 'Line 10 uses a simple token generatio

<IPython.core.display.Markdown object>